In [ ]:
!pip install fiftyone

In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F
import shutil
import os
import random

import fiftyone.zoo as foz

In [ ]:
def download_dataset():
    dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        label_types=["detections"],
        classes=["Elephant", "Tiger", "Deer", "Monkey", "Bear"],
        max_samples=2000,
        dataset_name="open-images-wildlife-2000",
        drop_existing_dataset=True
    )
    return dataset

In [ ]:
dataset = download_dataset()
dataset.persistent = True
dataset.save()

print(f"Dataset '{dataset.name}' saved and is now persistent.")
print(dataset)

In [ ]:
def filter_dataset(dataset, target_classes=[]):
    filtered_view = dataset.filter_labels(
        "ground_truth",
        F("label").is_in(target_classes)
    )
    
    if fo.dataset_exists("wildlife_clean_2000"):
        fo.delete_dataset("wildlife_clean_2000")

    # Clone filtered view properly
    clean_dataset = filtered_view.clone("wildlife_clean_2000")

    print("Clean dataset created:", clean_dataset.name)
    print("Class distribution:")
    print(clean_dataset.count_values("ground_truth.detections.label"))
    print("Original samples:", len(dataset))

    # Find samples with missing files
    missing_ids = []

    for sample in clean_dataset:
        if not os.path.exists(sample.filepath):
            missing_ids.append(sample.id)

    print("Missing files found:", len(missing_ids))

    # Delete broken samples
    if missing_ids:
        clean_dataset.delete_samples(missing_ids)
        print("Broken samples removed")

    # Save cleaned dataset
    clean_dataset.persistent = True
    clean_dataset.save()

    print("Remaining samples:", len(clean_dataset))
    
    return clean_dataset

In [ ]:
target_classes = ["Elephant", "Tiger", "Deer", "Monkey", "Bear"]
dataset = fo.load_dataset("open-images-wildlife-2000")
clean_dataset = filter_dataset(dataset, target_classes)

In [ ]:
def export_in_yolo_format(dataset, export_dir):
    if os.path.exists(export_dir):
        shutil.rmtree(export_dir)

    # Export to YOLOv5 format (YOLOv8 compatible)
    dataset.export(
        export_dir=export_dir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
    )

    print("Export complete")
    print("Total exported samples:", len(dataset))

In [ ]:
export_dir = "../../data/dataset/dataset_yolo"
export_in_yolo_format(clean_dataset, export_dir)

In [ ]:
def train_test_split_dataset(source_images_dir, source_labels_dir, destination_dir):
    
    splits = ["train", "val", "test"]

    # Create directories
    for split in splits:
        os.makedirs(f"{destination_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{destination_dir}/{split}/labels", exist_ok=True)

    images = os.listdir(source_images_dir)
    random.shuffle(images)

    train_split = int(0.7 * len(images))
    val_split = int(0.9 * len(images))

    train = images[:train_split]
    val = images[train_split:val_split]
    test = images[val_split:]

    def copy_files(files, split):
        for file in files:
            shutil.copy(
                os.path.join(source_images_dir, file),
                os.path.join(destination_dir, split, "images", file)
            )

            label = file.replace(".jpg", ".txt")
            shutil.copy(
                os.path.join(source_labels_dir, label),
                os.path.join(destination_dir, split, "labels", label)
            )

    copy_files(train, "train")
    copy_files(val, "val")
    copy_files(test, "test")

    print("Dataset split completed successfully.")